In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
from dataclasses import dataclass
import numpy as np
from typing import Any, List, Callable
from collections import defaultdict
from mdp import Policy, MDP, Step, value_evaluation, Rollout, FlackyTramMDP
from functools import partial

class RLAlgorithm:
    def get_action(self, state: Any) -> Any:
        raise NotImplementedError
    def incorporate_feedback(self, state: Any, action: Any, reward: float, next_state: Any, is_end: bool):
        raise NotImplementedError

class StaticAgent(RLAlgorithm):
    def __init__(self, policy: Policy):
        self.policy = policy

    def get_action(self, state: Any) -> Any:
        return self.policy(state)

    def incorporate_feedback(self, state: Any, action: Any, reward: float, next_state: Any, is_end: bool):
        pass

In [ ]:
def sample_transition(mdp: MDP, state: Any, action: Any) -> Step:
    steps = [successor for successor in mdp.successors(state) if successor.action == action]
    probs = [step.prob for step in steps]
    choice = np.random.choice(len(steps), p=probs)
    return steps[choice]

# Simulate multiple rollouts with an agent and MDP
# Policy: provided by agent
# Rollout environment: provided by MDP
# Return: utility of each rollout
def simulate(mdp: MDP, rl: RLAlgorithm, num_trials: int = 20) -> List[float]:
    utilities = []
    for _ in range(num_trials):
        state = mdp.start_state()
        steps = []
        while not mdp.is_end(state):
            action = rl.get_action(state)
            step = sample_transition(mdp, state, action)
             # !!end state should be next state in stead of state
            rl.incorporate_feedback(state, action, step.reward, step.state, mdp.is_end(step.state))
            steps.append(step)
            state = step.state
        rollout = Rollout(steps, mdp.discount())
        utilities.append(rollout.utility)
    return utilities

# Test simulation with static agent
tram_mdp = FlackyTramMDP(num_locs=10, failure_prob=0.4)
tram_policy = lambda s: "tram" if s * 2 <= tram_mdp.num_locs else "walk"
walk_policy = lambda s: "walk"
walk_agent = StaticAgent(walk_policy)
tram_agent = StaticAgent(tram_policy)
walk_utilities = simulate(tram_mdp, walk_agent, num_trials=1)
tram_utilities = simulate(tram_mdp, tram_agent, num_trials=30)
print("discount:", tram_mdp.discount())
print("walk utilities: ", walk_utilities)
print("tram utilities: ", tram_utilities)

discount: 0.9
walk utilities:  [-6.12579511]
tram utilities:  [-6.8051, -6.8051, -10.3809179, -6.8051, -9.312131, -6.8051, -6.8051, -9.312131, -12.987689149100001, -8.124590000000001, -8.124590000000001, -8.124590000000001, -12.208543499000001, -12.987689149100001, -11.34282611, -10.3809179, -12.208543499000001, -10.3809179, -12.208543499000001, -11.34282611, -9.312131, -8.124590000000001, -8.124590000000001, -9.312131, -9.312131, -6.8051, -8.124590000000001, -11.34282611, -8.124590000000001, -8.124590000000001]


In [13]:
#================== Model-Based Value Iteration ====================

class EstimateMDP(MDP):
    def __init__(self, discount: float):
        self.discount_ = discount
        # only one start state
        self.start_state_ = None
        # map from (state, action, next_state) to reward
        self.rewards = defaultdict(float)
        # map from (state, action, next_state) to count
        # { state: {
        #       action: {
        #          next_state: count
        #       }
        # }}
        self.transitions = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
        # support multiple end states
        self.end_states = set()

    def start_state(self) -> Any:
        return self.start_state_
        
    def successors(self, state: Any) -> List[Step]:
        steps = []
        for action, states in self.transitions[state].items():
            total_count = sum(states.values())
            for next_state, count in states.items():
                prob = count / total_count
                reward = self.rewards[(state, action, next_state)]
                steps.append(Step(action, reward, prob, next_state))
        return steps
    
    def is_end(self, state: Any) -> bool:
        return state in self.end_states
    
    def discount(self) -> float:
        return self.discount_

    def incorporate_feedback(self, state: Any, action: Any, reward: float, next_state: Any, is_end: bool):
        if self.start_state_ is None:
            self.start_state_ = state
        if is_end:
            # !!end state should be next state in stead of state
            self.end_states.add(next_state)
        self.rewards[(state, action, next_state)] = reward
        self.transitions[state][action][next_state] += 1

class ModelBasedValueIterationAgent(RLAlgorithm):
    def __init__(self, exploration_policy: Policy, discount: float = 1.0):
        # use exploration policy before estimated MDP is generated
        self.exploration_policy = exploration_policy
        # use estimated MDP as exploitation policy if exists
        self.exploitation_plicy = None
        self.mdp = EstimateMDP(discount)

    def get_action(self, state: Any) -> Any:
        if self.exploitation_plicy is None:
            return self.exploration_policy(state)
        else:
            # use exploration_policy as fallback policy if current state doesn't exist in MDP
            return self.exploitation_plicy.get(state, self.exploration_policy(state))
    
    def incorporate_feedback(self, state: Any, action: Any, reward: float, next_state: Any, is_end: bool):
        self.mdp.incorporate_feedback(state, action, reward, next_state, is_end)

    # Run value iteration to compute the optimal policy for the estimated MDP
    # use MDP as exploitation policy only after this call, usually after a couple of rollouts with exploration policy
    def run_value_iteration(self):
        self.exploitation_plicy = value_evaluation(self.mdp).pi

def walk_tram_exploration_policy(num_locs: int, state: Any) -> Any:
    if state * 2 <= num_locs:
        return np.random.choice(["walk", "tram"])
    else:
        return "walk"

# Test model based value iteration
exploration_policy = partial(walk_tram_exploration_policy, tram_mdp.num_locs)
agent = ModelBasedValueIterationAgent(exploration_policy, tram_mdp.discount())
exploration_utilities = simulate(tram_mdp, agent, 10)
print(np.mean(exploration_utilities))
agent.run_value_iteration()
value_iteration_utilities = simulate(tram_mdp, agent, 10)
print("optimal policy:", agent.exploitation_plicy)
print(np.mean(value_iteration_utilities))


-6.744193211999999
optimal policy: {1: np.str_('walk'), 2: np.str_('walk'), 3: np.str_('walk'), 4: np.str_('walk'), 5: np.str_('tram'), 6: 'walk', 7: 'walk', 8: 'walk', 9: 'walk', 10: None}
-5.3298802
